# 02. Data Pre-Cleaning

Transforms raw data into a pre-cleaned state for downstream processing. Each step logs changes.

In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import re

pd.set_option("mode.copy_on_write", False)

In [ ]:
raw_dir = Path.cwd().parent / "data" / "raw"
proc_dir = Path.cwd().parent / "data" / "processed"
proc_dir.mkdir(parents=True, exist_ok=True)

csv_files = list(raw_dir.glob("*.csv"))
if not csv_files:
    raise FileNotFoundError(f"Place a CSV in {raw_dir}")

file_path = csv_files[0]
df = pd.read_csv(file_path, encoding="utf-8", low_memory=False)
n_row0, n_col0 = df.shape
print(f"Loaded: {file_path.name}  ({n_row0:,} rows x {n_col0:,} cols)")

## 1. Drop high-null columns

Remove columns where more than 40% of values are missing.

In [ ]:
null_pct = df.isna().mean() * 100
high_null_cols = null_pct[null_pct > 40].index.tolist()
df.drop(columns=high_null_cols, inplace=True)
print(f"Dropped {len(high_null_cols)} column(s) with >40% nulls: {high_null_cols}")

## 2. Drop constant and duplicate columns

In [ ]:
# Constant columns (single value)
const_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
df.drop(columns=const_cols, inplace=True)
print(f"Dropped {len(const_cols)} constant column(s): {const_cols}")

In [ ]:
# Duplicate columns (identical values)
dup_cols = []
seen = {}
for col in df.columns:
    key = tuple(df[col].astype(str).fillna("__NULL__"))
    if key in seen:
        dup_cols.append(col)
    else:
        seen[key] = col
df.drop(columns=dup_cols, inplace=True)
print(f"Dropped {len(dup_cols)} duplicate column(s): {dup_cols}")

## 3. Standardize column names

Convert to snake_case: lowercase, spaces/hyphens to underscores, strip special chars.

In [ ]:
def clean_col_name(name: str) -> str:
    name = name.strip().lower()
    name = re.sub(r"[^\w\s]", "", name)
    name = re.sub(r"[\s\-]+", "_", name)
    name = re.sub(r"_+", "_", name).strip("_")
    return name

old_names = list(df.columns)
new_names = [clean_col_name(c) for c in old_names]
rename_map = {o: n for o, n in zip(old_names, new_names) if o != n}
df.rename(columns=rename_map, inplace=True)
print(f"Renamed {len(rename_map)} column(s)")
if rename_map:
    for old, new in rename_map.items():
        print(f"  {old:30s} -> {new}")

## 4. String cleaning

Strip leading/trailing whitespace from all object columns.

In [ ]:
str_cols = df.select_dtypes(include=["object"]).columns
for col in str_cols:
    df[col] = df[col].astype(str).str.strip().replace("nan", None)
print(f"Cleaned {len(str_cols)} string column(s)")

## 5. Type conversion

Attempt to parse dates, coerce numeric strings, and optimize categoricals.

In [ ]:
# Try date parsing on object columns
date_cols = []
for col in df.select_dtypes(include=["object"]).columns:
    try:
        parsed = pd.to_datetime(df[col], format="mixed", errors="coerce")
        if parsed.notna().sum() > len(df) * 0.5:
            df[col] = parsed
            date_cols.append(col)
    except Exception:
        pass
print(f"Parsed {len(date_cols)} date column(s): {date_cols}")

In [ ]:
# Attempt numeric coercion on remaining object columns
num_coerced = []
for col in df.select_dtypes(include=["object"]).columns:
    coerced = pd.to_numeric(df[col], errors="coerce")
    if coerced.notna().sum() > len(df) * 0.5:
        df[col] = coerced
        num_coerced.append(col)
print(f"Coerced {len(num_coerced)} column(s) to numeric: {num_coerced}")

In [ ]:
# Optimize low-cardinality categoricals
cat_cols = df.select_dtypes(include=["object"]).columns
cat_opt = []
for col in cat_cols:
    if df[col].nunique() / len(df) < 0.05:
        df[col] = df[col].astype("category")
        cat_opt.append(col)
print(f"Optimized {len(cat_opt)} column(s) to category: {cat_opt}")

## 6. Deduplicate rows

In [ ]:
n_before = len(df)
df.drop_duplicates(inplace=True)
n_dupes = n_before - len(df)
print(f"Removed {n_dupes:,} duplicate row(s) ({n_dupes/n_before*100:.2f}%)")

## 7. Save pre-cleaned data

In [ ]:
out_name = f"precleaned_{file_path.stem}.csv"
out_path = proc_dir / out_name
df.to_csv(out_path, index=False)
print(f"Saved: {out_path}")

## 8. Pre-cleaning Report

In [ ]:
n_row1, n_col1 = df.shape
print("=" * 55)
print("  PRE-CLEANING REPORT")
print("=" * 55)
print(f"  Original shape:           {n_row0:>8,} rows x {n_col0:,} cols")
print(f"  Final shape:              {n_row1:>8,} rows x {n_col1:,} cols")
print(f"  Rows removed (dupes):     {n_dupes:>8,}")
print(f"  Columns dropped:          {n_col0 - n_col1:>8}")
print(f"    - High null (>40%):     {len(high_null_cols):>8}")
print(f"    - Constant:             {len(const_cols):>8}")
print(f"    - Duplicate:            {len(dup_cols):>8}")
print(f"  Columns renamed:          {len(rename_map):>8}")
print(f"  Dates parsed:             {len(date_cols):>8}")
print(f"  Coerced to numeric:       {len(num_coerced):>8}")
print(f"  Optimized to category:    {len(cat_opt):>8}")
print(f"  Remaining missing:        {df.isna().sum().sum():>8,}")